# ECG Signal Processing — Part 1

This notebook processes an ECG (electrocardiogram) time-series signal through a full pipeline:
1. Load and inspect the raw signal
2. Handle missing values via interpolation
3. Denoise using Fast Fourier Transform (low-pass filter at 100 Hz)
4. Detect R-peaks using cross-correlation with custom kernels
5. Calculate instantaneous heart rate from R-peak intervals

**Tools:** NumPy, SciPy (FFT, ndimage), Pandas, Matplotlib

In [ ]:
#library imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import fft
from scipy.ndimage import correlate
import pandas as pd

### Step 1: Load data and plot the signal

In [ ]:
data = pd.read_csv('project_ecg_part1_signal_missing_value.csv') # update this path to match your local data file

In [ ]:
x=data['Amp'].values
t=data['t'].values

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(t, x)
plt.title('ECG signal x (noisy)')
plt.xlabel('Time [s]')
plt.ylabel('Signal Amplitude')
plt.grid(True)
plt.show()

In [ ]:
dt = t[1] - t[0]
print("the sampling interval is", dt, "seconds")

In [ ]:
fs = 1/dt
print("the sampling frequency is",fs,"Hz")

### Step 2: Handle missing values

In [ ]:
data.isnull().sum()

The above results show that there are five missing values in the column `Amp`

In [ ]:
data['Amp'] = data['Amp'].interpolate()

In [ ]:
x = data['Amp'].values

### Step 3: Noise reduction via Fourier Transform

In [ ]:
N = len(x)
dt = t[1] - t[0]
fs = 1 / dt

X = fft.fft(x)
X_abs = np.abs(X)
freqs = fft.fftfreq(N, d=dt)

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(freqs, X_abs)
plt.title('Magnitude Spectrum')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude of Frequency Component')
plt.xlim(-300, 300)
plt.grid(True)
plt.show()

The spectrum shows high-frequency components above 100 Hz and below -100 Hz — these are noise. We zero them out using NumPy boolean indexing, then apply the inverse FFT to recover a clean signal.

In [ ]:
X[(freqs < -100) | (freqs > 100)] = 0

In [ ]:
x_clean = np.real(fft.ifft(X))

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(t, x_clean)
plt.title('ECG signal x_clean (after FFT denoising)')
plt.xlabel('Time [s]')
plt.ylabel('Signal Amplitude')
plt.grid(True)
plt.show()

### Step 4: R-Peak Detection via cross-correlation

Two derivative-based kernels are convolved with the clean signal to identify upward zero-crossings (R-peaks):
- `h1 = [-1, 1, 0]` — detects rising edge
- `h2 = [0, 1, -1]` — detects falling edge

A sample is flagged as a peak where both responses are positive. R-peaks are then isolated by applying an amplitude threshold of `mean + 2 × std`.

In [ ]:
h1 = [-1, 1, 0]
h2 = [0, 1, -1]

y = correlate(x_clean, h1, mode='nearest')
z = correlate(x_clean, h2, mode='nearest')

PeakIndexArray = np.where((y > 0) & (z > 0))[0]

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(t, x_clean, label='x_clean')
plt.plot(t[PeakIndexArray], x_clean[PeakIndexArray], 'o', label='peaks')
plt.title('ECG signal x_clean and detected peaks')
plt.xlabel('Time [s]')
plt.ylabel('Signal Amplitude')
plt.legend()
plt.grid(True)
plt.show()

R-peaks are the dominant peaks in each cardiac cycle. We isolate them by keeping only peaks above a threshold of `mean + 2 × std` of all detected peak amplitudes.

In [ ]:
peak_vals = x_clean[PeakIndexArray]
threshold = np.mean(peak_vals) + 2 * np.std(peak_vals)
print("threshold is", threshold)

In [ ]:
RPeakIndexArray = PeakIndexArray[x_clean[PeakIndexArray] > threshold]

plt.figure(figsize=(10,4))
plt.plot(t, x_clean, label='x_clean')
plt.plot(t[RPeakIndexArray], x_clean[RPeakIndexArray], 'o', label='R-peaks', color='orange')
plt.title('ECG signal x_clean and R-peaks')
plt.xlabel('Time [s]')
plt.ylabel('Signal Amplitude')
plt.legend()
plt.grid(True)
plt.show()

### Step 5: Calculate heart rate

In [ ]:
HeartRate = np.zeros(RPeakIndexArray.shape)

for n in range(1, RPeakIndexArray.shape[0]):
    HeartRate[n] = 60 * fs / (RPeakIndexArray[n] - RPeakIndexArray[n - 1])

HeartRate[0] = HeartRate[1]

fig, axs = plt.subplots(2, 1, figsize=(10, 6))

axs[0].plot(t, x_clean)
axs[0].plot(t[RPeakIndexArray], x_clean[RPeakIndexArray], 'o', color='orange')
axs[0].set_title('ECG signal x_clean and R-peaks')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Signal Amplitude')

axs[1].plot(t[RPeakIndexArray], HeartRate)
axs[1].set_title('Heart Rate (beats per minute)')
axs[1].set_xlabel('Time [s]')
axs[1].set_ylabel('Heart Rate (BPM)')
axs[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()